# 03 · Modeling (CRISP-DM step 4)

**Run this on Google Colab with a GPU:** Runtime → Change runtime type → T4 GPU.

Three experiments, all on the same train/validation split and judged on validation data
(the test set stays untouched until notebook 04):

| # | Model | Why |
|---|---|---|
| 0 | Small CNN from scratch | Baseline: shows what is possible without pretrained knowledge |
| 1 | MobileNetV2, frozen + new head | Transfer learning: reuse ImageNet features, train only the classifier |
| 2 | MobileNetV2, top 40 layers fine-tuned | Let the top layers adapt to coffee beans, with a tiny learning rate |

**Outputs:** models in `05_modeling/models/`, epoch logs in `06_evaluation/metrics/`,
training curves in `reports/figures/` (prefix `03_`).

In [ ]:
# --- SETUP: run this cell first -----------------------------------------
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Change this if you uploaded the project folder somewhere else in your Drive:
    PROJECT = Path("/content/drive/MyDrive/kapeklasipika")
    %pip install -q imagehash
else:
    # Running on your laptop: walk up from this notebook until we find config/config.yaml
    PROJECT = Path.cwd()
    while not (PROJECT / "config" / "config.yaml").exists():
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.config import load_config, use_local_copy
cfg = load_config()
print("Project folder:", PROJECT)

In [ ]:
# Colab only: copy the images to Colab's fast local disk (Drive is slow for 8,000 files)
if IN_COLAB:
    cfg = use_local_copy(cfg)

In [ ]:
import time
import pandas as pd
import tensorflow as tf
from src.data_loading import load_all_splits
from src.models import build_baseline_cnn, build_mobilenetv2, unfreeze_top_layers, make_callbacks, count_params
from src.evaluate import plot_history

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow)")
tf.keras.utils.set_random_seed(cfg["seed"])   # same random choices every run -> reproducible

T = cfg["training"]; SIZE = cfg["img_size"]
MODELS = cfg["paths"]["models"]; METRICS = cfg["paths"]["metrics"]; FIG = cfg["paths"]["figures"]
for d in (MODELS, METRICS, FIG):
    d.mkdir(parents=True, exist_ok=True)

train_ds, val_ds, test_ds = load_all_splits(cfg)

## Experiment 0 · Baseline CNN
An *epoch* is one full pass over the 4,800 training images.
Early stopping ends training once validation loss stops improving for 5 epochs.

In [ ]:
log0 = METRICS / "history_baseline.csv"; log0.unlink(missing_ok=True)
baseline = build_baseline_cnn(SIZE, lr=T["head_lr"], seed=cfg["seed"])
print(count_params(baseline))

t0 = time.time()
baseline.fit(train_ds, validation_data=val_ds, epochs=T["baseline_epochs"],
             callbacks=make_callbacks(MODELS / "baseline_cnn.keras", log0, T["patience"]))
print(f"Training time: {(time.time() - t0) / 60:.1f} min")
plot_history(log0, "Baseline CNN", FIG / "03_history_baseline.png");

## Experiment 1 · MobileNetV2, frozen base
Only the new layers on top (pooling, dropout, 4-class output) are trained.

In [ ]:
log1 = METRICS / "history_mobilenetv2.csv"; log1.unlink(missing_ok=True)
mnv2, base = build_mobilenetv2(SIZE, lr=T["head_lr"], seed=cfg["seed"])
print(count_params(mnv2))

t0 = time.time()
h1 = mnv2.fit(train_ds, validation_data=val_ds, epochs=T["head_epochs"],
              callbacks=make_callbacks(MODELS / "mobilenetv2_head.keras", log1, T["patience"]))
print(f"Training time: {(time.time() - t0) / 60:.1f} min")
stage1_epochs = len(h1.history["loss"])

## Experiment 2 · MobileNetV2, fine-tuned
We unfreeze the last 40 layers of MobileNetV2 and continue with a learning rate 100× smaller,
so the pretrained features are adjusted gently instead of overwritten.
The epoch log continues in the same file, so the curve shows both stages.

In [ ]:
mnv2 = unfreeze_top_layers(mnv2, base, n_layers=T["finetune_layers"], lr=T["finetune_lr"])
print(count_params(mnv2))

t0 = time.time()
mnv2.fit(train_ds, validation_data=val_ds,
         epochs=stage1_epochs + T["finetune_epochs"], initial_epoch=stage1_epochs,
         callbacks=make_callbacks(MODELS / "mobilenetv2_finetuned.keras", log1, T["patience"]))
print(f"Training time: {(time.time() - t0) / 60:.1f} min")
plot_history(log1, f"MobileNetV2 (fine-tuning starts after epoch {stage1_epochs})", FIG / "03_history_mobilenetv2.png");

## Validation summary
Compare the three models on the **validation** set. The best one goes to notebook 04.
Copy this table into `05_modeling/experiments_log.md`.

In [ ]:
rows = []
for name in ["baseline_cnn", "mobilenetv2_head", "mobilenetv2_finetuned"]:
    m = tf.keras.models.load_model(MODELS / f"{name}.keras")
    tr_loss, tr_acc = m.evaluate(train_ds, verbose=0)
    va_loss, va_acc = m.evaluate(val_ds, verbose=0)
    rows.append({"model": name, "train_acc": round(tr_acc, 4), "val_acc": round(va_acc, 4),
                 "gap": round(tr_acc - va_acc, 4), "val_loss": round(va_loss, 4), **count_params(m)})
summary = pd.DataFrame(rows)
summary.to_csv(METRICS / "val_summary.csv", index=False)
summary

**How to read `gap`:** train accuracy minus validation accuracy. A large gap (say over 0.10)
means overfitting. The original paper's gap was about 0.17 (98.8% vs 81.3%).

If you used Colab, the models are already saved to your Drive project folder.